In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from OA_utils.data_utils import *
import random
import pickle

Load data from file

In [2]:
data_dir      = 'C:\\Users\\bakel\\Desktop\\GRFMuscleModel\\data\\'
filename      = 'resampled_segments.pkl'   # <-- change to your file
output_prefix = 'Ulrich_jrf_cop_leg'                      # <-- prefix for saved .npz files

SKIP_SUBJECTS = {'time_resampled', 'OA11'}  # subjects to exclude

# Set True to package the entire dataset as a test set only (no train/val split).
# Useful for cross-testing models trained on a different dataset.
TEST_ONLY = False

with open(data_dir + filename, 'rb') as f:
    segs = pickle.load(f)

print('Loaded', filename)

Loaded resampled_segments.pkl


Visualize amount of data present for each subject

In [3]:
def count_total_segments(split_dict, key='grf_y'):
    return sum(len(split_dict[subj][key]) for subj in split_dict)

subjects   = [s for s in segs.keys() if s not in SKIP_SUBJECTS]
total_segs = 0
for subj in subjects:
    n = len(segs[subj]['grf_y'])
    total_segs += n
    print(f'{subj}: {n} segments')
print(total_segs, 'segments total')

Subject1: 1357 segments
Subject2: 1517 segments
Subject3: 1526 segments
Subject4: 1338 segments
Subject5: 1482 segments
Subject6: 1544 segments
Subject7: 1539 segments
Subject8: 1581 segments
Subject9: 1450 segments
Subject10: 1558 segments
14892 segments total


## Split
If `TEST_ONLY = True`, the entire dataset becomes the test set and the seed search is skipped.

In [4]:
if TEST_ONLY:
    train_dict = {}
    val_dict   = {}
    test_dict  = {s: segs[s] for s in subjects}
    print(f'TEST_ONLY mode — all {total_segs} segments → test set')

else:
    best_seed = None
    best_err  = float('inf')
    best_counts = []
    target_train, target_val, target_test = 0.8, 0.1, 0.1

    for seed in range(40):
        random.seed(seed)
        shuf = subjects.copy()
        random.shuffle(shuf)

        N         = len(shuf)
        train_end = int(N * 0.8)
        val_end   = int(N * 0.9)

        td = {s: segs[s] for s in shuf[:train_end]}
        vd = {s: segs[s] for s in shuf[train_end:val_end]}
        ed = {s: segs[s] for s in shuf[val_end:]}

        tc = count_total_segments(td)
        vc = count_total_segments(vd)
        ec = count_total_segments(ed)

        err = (abs(tc/total_segs - target_train) +
               abs(vc/total_segs - target_val)   +
               abs(ec/total_segs - target_test))

        if err < best_err:
            best_err    = err
            best_seed   = seed
            best_counts = (tc, vc, ec)

        print('Best seed:', best_seed,
              ' counts:', best_counts,
              ' error:', round(best_err, 6))

    # Rebuild with best seed
    random.seed(best_seed)
    shuf = subjects.copy()
    random.shuffle(shuf)
    N         = len(shuf)
    train_end = int(N * 0.8)
    val_end   = int(N * 0.9)

    train_dict = {s: segs[s] for s in shuf[:train_end]}
    val_dict   = {s: segs[s] for s in shuf[train_end:val_end]}
    test_dict  = {s: segs[s] for s in shuf[val_end:]}

    print(f'\nBest seed: {best_seed}   ratios: {best_counts[0]/total_segs:.2f} / {best_counts[1]/total_segs:.2f} / {best_counts[2]/total_segs:.2f}')
    print('Train:', list(train_dict.keys()), '->', count_total_segments(train_dict), 'segs')
    print('Val:  ', list(val_dict.keys()),   '->', count_total_segments(val_dict),   'segs')
    print('Test: ', list(test_dict.keys()),  '->', count_total_segments(test_dict),  'segs')

Best seed: 0  counts: (11795, 1558, 1539)  error: 0.015928
Best seed: 1  counts: (11849, 1517, 1526)  error: 0.008676
Best seed: 1  counts: (11849, 1517, 1526)  error: 0.008676
Best seed: 1  counts: (11849, 1517, 1526)  error: 0.008676
Best seed: 1  counts: (11849, 1517, 1526)  error: 0.008676
Best seed: 1  counts: (11849, 1517, 1526)  error: 0.008676
Best seed: 1  counts: (11849, 1517, 1526)  error: 0.008676
Best seed: 1  counts: (11849, 1517, 1526)  error: 0.008676
Best seed: 1  counts: (11849, 1517, 1526)  error: 0.008676
Best seed: 1  counts: (11849, 1517, 1526)  error: 0.008676
Best seed: 1  counts: (11849, 1517, 1526)  error: 0.008676
Best seed: 1  counts: (11849, 1517, 1526)  error: 0.008676
Best seed: 1  counts: (11849, 1517, 1526)  error: 0.008676
Best seed: 1  counts: (11849, 1517, 1526)  error: 0.008676
Best seed: 14  counts: (11925, 1450, 1517)  error: 0.005265
Best seed: 14  counts: (11925, 1450, 1517)  error: 0.005265
Best seed: 14  counts: (11925, 1450, 1517)  error: 0.0

In [5]:
print(segs['Subject1'].keys())

dict_keys(['grf_x', 'grf_y', 'grf_z', 'cop_x', 'cop_y', 'cop_z', 'addbrev', 'addlong', 'addmagDist', 'addmagIsch', 'addmagMid', 'addmagProx', 'bflh', 'bfsh', 'edl', 'ehl', 'fdl', 'fhl', 'gaslat', 'gasmed', 'glmax1', 'glmax2', 'glmax3', 'glmed1', 'glmed2', 'glmed3', 'glmin1', 'glmin2', 'glmin3', 'grac', 'iliacus', 'perbrev', 'perlong', 'psoas', 'recfem', 'semimem', 'semiten', 'soleus', 'tibant', 'tibpost', 'vasint', 'vaslat', 'vasmed', 'knee_fx', 'knee_fy', 'knee_fz', 'ankle_fx', 'ankle_fy', 'ankle_fz', 'achilles'])


In [6]:
def get_subject_ranges(split_dict):
    ranges = {}
    start_idx = 0
    for subj, data in split_dict.items():
        n = len(data['grf_x'])
        end_idx = start_idx + n
        ranges[subj] = (start_idx, end_idx)
        start_idx = end_idx
    return ranges

print('Test subject ranges:', get_subject_ranges(test_dict))

Test subject ranges: {'Subject5': (0, 1482)}


## Signal definitions — update here when adding new inputs/outputs

In [7]:
INPUT_KEYS  = ['grf_x', 'grf_y', 'grf_z', 'cop_x', 'cop_y', 'cop_z']
OUTPUT_KEYS = [ 'addbrev', 'addlong', 'addmagDist', 'addmagIsch', 'addmagMid', 
               'addmagProx', 'bflh', 'bfsh', 'edl', 'ehl', 'fdl', 'fhl', 'gaslat', 
               'gasmed', 'glmax1', 'glmax2', 'glmax3', 'glmed1', 'glmed2', 'glmed3', 
               'glmin1', 'glmin2', 'glmin3', 'grac', 'iliacus', 'perbrev', 'perlong', 
               'psoas', 'recfem', 'semimem', 'semiten', 'soleus', 'tibant', 'tibpost', 
               'vasint', 'vaslat', 'vasmed', 'achilles', 'knee_fx', 'knee_fy', 'knee_fz', 
               'ankle_fx', 'ankle_fy', 'ankle_fz'
]
ALL_KEYS  = INPUT_KEYS + OUTPUT_KEYS
N_INPUTS  = len(INPUT_KEYS)
N_OUTPUTS = len(OUTPUT_KEYS)

print(f'Inputs  ({N_INPUTS}):  {INPUT_KEYS}')
print(f'Outputs ({N_OUTPUTS}): {OUTPUT_KEYS}')

sample_keys = set(segs[subjects[0]].keys())
missing  = [k for k in ALL_KEYS if k not in sample_keys]
uncaught = [k for k in sample_keys if k not in ALL_KEYS]

if missing:
    print(f"ERROR — keys defined but NOT in data:    {missing}")
else:
    print("OK — all INPUT_KEYS + OUTPUT_KEYS present in data.")

if uncaught:
    print(f"INFO  — keys in data but NOT in ALL_KEYS: {uncaught}")


Inputs  (6):  ['grf_x', 'grf_y', 'grf_z', 'cop_x', 'cop_y', 'cop_z']
Outputs (44): ['addbrev', 'addlong', 'addmagDist', 'addmagIsch', 'addmagMid', 'addmagProx', 'bflh', 'bfsh', 'edl', 'ehl', 'fdl', 'fhl', 'gaslat', 'gasmed', 'glmax1', 'glmax2', 'glmax3', 'glmed1', 'glmed2', 'glmed3', 'glmin1', 'glmin2', 'glmin3', 'grac', 'iliacus', 'perbrev', 'perlong', 'psoas', 'recfem', 'semimem', 'semiten', 'soleus', 'tibant', 'tibpost', 'vasint', 'vaslat', 'vasmed', 'achilles', 'knee_fx', 'knee_fy', 'knee_fz', 'ankle_fx', 'ankle_fy', 'ankle_fz']
OK — all INPUT_KEYS + OUTPUT_KEYS present in data.


In [8]:
def dict_to_array(split_dict):
    packed_segments = []
    for subj, data in split_dict.items():
        num_segs = len(data[INPUT_KEYS[0]])
        for i in range(num_segs):
            sample = np.column_stack([data[k][i] for k in ALL_KEYS])
            packed_segments.append(sample)
    return np.array(packed_segments)

if TEST_ONLY is False:
    train_arr = dict_to_array(train_dict)
    val_arr   = dict_to_array(val_dict)
    print("Train:", train_arr.shape)
    print("Val:",   val_arr.shape)
else:
    train_arr = None
    val_arr = None

test_arr  = dict_to_array(test_dict)
print("Test:",  test_arr.shape)

Train: (11893, 100, 50)
Val: (1517, 100, 50)
Test: (1482, 100, 50)


In [9]:
if TEST_ONLY is False:
    X_train, y_train = train_arr[:, :, :N_INPUTS], train_arr[:, :, N_INPUTS:]
    X_val,   y_val   = val_arr[:,   :, :N_INPUTS], val_arr[:,   :, N_INPUTS:]
    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"X_val shape:   {X_val.shape}")
    print(f"y_val shape:   {y_val.shape}")
X_test,  y_test  = test_arr[:,  :, :N_INPUTS], test_arr[:,  :, N_INPUTS:]


print(f"X_test shape:  {X_test.shape}")
print(f"y_test shape:  {y_test.shape}")

X_train shape: (11893, 100, 6)
y_train shape: (11893, 100, 44)
X_val shape:   (1517, 100, 6)
y_val shape:   (1517, 100, 44)
X_test shape:  (1482, 100, 6)
y_test shape:  (1482, 100, 44)


In [ ]:
if TEST_ONLY is False:
    np.savez(data_dir + f'{output_prefix}_train_data.npz', X_train=X_train, y_train=y_train)
    np.savez(data_dir + f'{output_prefix}_val_data.npz',   X_val=X_val,     y_val=y_val)
np.savez(data_dir + f'{output_prefix}_test_data.npz',  X_test=X_test,   y_test=y_test, output_keys = OUTPUT_KEYS)